In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("matplotlib:", plt.matplotlib.__version__)

import sklearn
print("matplotlib:", plt.matplotlib.__version__)

import xgboost as xgb
print("xgboost:", xgb.__version__)

import mapie
print("mapie:", mapie.__version__)

DATA_DIR = Path("../data/raw")
print("\nData dir exists:", DATA_DIR.exists())
print("files found:")
for f in DATA_DIR.iterdir():
    print(" ", f.name)

pandas: 2.3.1
numpy: 1.26.4
matplotlib: 3.9.4
matplotlib: 3.9.4
xgboost: 2.1.4
mapie: 1.0.1

Data dir exists: True
files found:
  41586_2018_623_MOESM3_ESM.xlsx
  aml_ohsu_2022
  beataml_waves1to4_sample_mapping.xlsx


In [21]:
#  load and inspect the expression file

AML_DIR = DATA_DIR / "aml_ohsu_2022"

expr_path = AML_DIR / "data_mrna_seq_rpkm_zscores_ref_all_samples.txt"
expr_raw = pd.read_csv(expr_path, sep="\t", nrows=5)
print("Shape (5 rows preview):", expr_raw.shape)
print(expr_raw.iloc[:, :6])
print("\nFirst 3 column names:", expr_raw.columns[:3].tolist())
print("Last 3 column names:", expr_raw.columns[-3:].tolist())

Shape (5 rows preview): (5, 672)
  Hugo_Symbol  aml_ohsu_2022_2157_BA2452  aml_ohsu_2022_2269_BA2922  \
0      TSPAN6                    -2.7865                    -3.0837   
1        DPM1                    -1.3511                    -1.2484   
2       SCYL3                    -0.2185                     1.4769   
3    C1orf112                     1.1468                    -0.7416   
4         FGR                    -1.9246                    -0.9194   

   aml_ohsu_2022_2313_BA2707  aml_ohsu_2022_2418_BA2409  \
0                    -4.8277                    -2.8720   
1                    -0.8005                    -0.1683   
2                    -1.3015                     0.0664   
3                     0.5396                    -0.8389   
4                    -0.0876                     1.1379   

   aml_ohsu_2022_2374_BA2480  
0                    -1.4848  
1                    -0.7805  
2                     0.8273  
3                    -0.6725  
4                    -0.0383  

In [22]:
expr_full = pd.read_csv(expr_path, sep="\t", index_col=0)
print("Raw shape (genes x patients):", expr_full.shape)

expr = expr_full.T
print("Transposed shape (patients × genes):", expr.shape)
print("Sample index preview:", expr.index[:3].tolist())
print("Missing values:", expr.isnull().sum().sum())

Raw shape (genes x patients): (22843, 671)
Transposed shape (patients × genes): (671, 22843)
Sample index preview: ['aml_ohsu_2022_2157_BA2452', 'aml_ohsu_2022_2269_BA2922', 'aml_ohsu_2022_2313_BA2707']
Missing values: 6039


In [23]:
# diagnose the missing values
missing_per_gene = expr.isnull().sum()
missing_genes = missing_per_gene[missing_per_gene > 0]

print(f"Genes with any missing values: {len(missing_genes)} / {expr.shape[1]}")
print(f"Max missing in one gene: {missing_genes.max()}")
print(f"Genes missing in >50% of patients: {(missing_genes > expr.shape[0]*0.5).sum()}")
print(f"Genes missing in >10% of patients: {(missing_genes > expr.shape[0]*0.1).sum()}")
print("\nDistribution of missingness:")
print(missing_genes.describe())

Genes with any missing values: 9 / 22843
Max missing in one gene: 671
Genes missing in >50% of patients: 9
Genes missing in >10% of patients: 9

Distribution of missingness:
count      9.0
mean     671.0
std        0.0
min      671.0
25%      671.0
50%      671.0
75%      671.0
max      671.0
dtype: float64


In [24]:
# drop the empty genes
empty_genes = missing_per_gene[missing_per_gene == expr.shape[0]].index.tolist()
print("Dropping these 9 genes:", empty_genes)

expr = expr.drop(columns=empty_genes)
print("\nClean expression matrix shape:", expr.shape)
print("Missing values remaining:", expr.isnull().sum().sum())

Dropping these 9 genes: ['DNAH11', 'SEC16B', 'UNC13A', 'CARD14', 'IQUB', 'ENPP1', 'LRRD1', 'WDR65', 'MROH7-TTC4']

Clean expression matrix shape: (671, 22834)
Missing values remaining: 0


In [25]:
# load and inspect the drug sensitivity data

drug_raw = pd.read_excel(drug_path, sheet_name="Table S10-Drug Responses")
print("Shape:", drug_raw.shape)
print("Columns:", drug_raw.columns.tolist())
print("\nFirst 5 rows:")
print(drug_raw.head())
print("\nUnique drugs:", drug_raw['inhibitor'].nunique() if 'inhibitor' in drug_raw.columns else "CHECK COLUMN NAME")
print("Unique patients:", drug_raw['lab_id'].nunique() if 'lab_id' in drug_raw.columns else "CHECK COLUMN NAME")

Shape: (47650, 4)
Columns: ['inhibitor', 'lab_id', 'ic50', 'auc']

First 5 rows:
               inhibitor    lab_id       ic50         auc
0  17-AAG (Tanespimycin)  12-00211  10.000000  225.918025
1  17-AAG (Tanespimycin)  12-00219   0.276661  135.264409
2  17-AAG (Tanespimycin)  12-00258   2.722845  164.561227
3  17-AAG (Tanespimycin)  12-00262   0.123136  111.555971
4  17-AAG (Tanespimycin)  12-00268  10.000000  226.805281

Unique drugs: 122
Unique patients: 528


In [26]:
mapping = pd.read_excel(DATA_DIR / "beataml_waves1to4_sample_mapping.xlsx")
print("Shape:", mapping.shape)
print("Columns:", mapping.columns.tolist())
print("\nFirst 5 rows:")
print(mapping.head())

Shape: (979, 12)
Columns: ['patientId', 'dbgap_subject_id', 'labId', 'dbgap_rnaseq_sample', 'rna_seq_id', 'rna_control', 'rna_include_in_analysis', 'dbgap_dnaseq_sample', 'dna_seq_id', 'dna_capture_type', 'dna_include_in_analysis', 'analysisDrug']

First 5 rows:
   patientId  dbgap_subject_id     labId dbgap_rnaseq_sample rna_seq_id  \
0          0              2096  00-00002             BA2392R   00-00002   
1          0              2096  00-00003             BA2611R   00-00003   
2          0              2096  00-00004             BA2506R   00-00004   
3          0              2096  00-00005             BA2430R   00-00005   
4          0              2096  00-00006             BA2448R   00-00006   

            rna_control rna_include_in_analysis dbgap_dnaseq_sample  \
0  Healthy pooled CD34+                     yes                 NaN   
1  Healthy pooled CD34+                     yes                 NaN   
2  Healthy pooled CD34+                     yes                 NaN   
3 

In [30]:
# check labId vs drug lab_id overlap directly

mapping_labids = set(mapping['labId'].dropna())
drug_labids = set(drug_raw['lab_id'].dropna())

overlap = mapping_labids & drug_labids
print("Direct labId overlap between mapping and drug data:", len(overlap))
print("Sample overlapping IDs:", list(overlap)[:5])

print("\nUnique prefixes in mapping labId:", mapping['labId'].dropna().str.extract(r'^(\d+)-')[0].unique())
print("Unique prefixes in drug lab_id:", drug_raw['lab_id'].str.extract(r'^(\d+)-')[0].unique())

Direct labId overlap between mapping and drug data: 417
Sample overlapping IDs: ['14-00329', '13-00262', '16-00867', '13-00237', '16-00611']

Unique prefixes in mapping labId: ['00' '10' '14' '13' '11' '12' '09' '15' '16' '17' '18' '19']
Unique prefixes in drug lab_id: ['12' '13' '14' '15' '16' '17' '11']


In [31]:
# build the full ID bridge and check final linkable cohort

id_bridge = mapping[['labId', 'dbgap_subject_id']].dropna(subset=['labId']).copy()
id_bridge['cbio_patient_id'] = 'aml_ohsu_2022_' + id_bridge['dbgap_subject_id'].astype(str)
id_bridge = id_bridge.rename(columns={'labId': 'lab_id'})
id_bridge = id_bridge.drop_duplicates(subset='lab_id')

drug_with_patient = drug_raw.merge(id_bridge[['lab_id', 'cbio_patient_id']], on='lab_id', how='inner')
print("Drug rows after mapping:", len(drug_with_patient))
print("Unique patients in drug data with patient ID:", drug_with_patient['cbio_patient_id'].nunique())
print("Unique drugs:", drug_with_patient['inhibitor'].nunique())

expr_patient_ids = set(expr.index.str.extract(r'(aml_ohsu_2022_\d+)')[0].dropna())
final_patients = set(drug_with_patient['cbio_patient_id']) & expr_patient_ids
print("\nPatients with both drug AND expression data:", len(final_patients))

Drug rows after mapping: 39137
Unique patients in drug data with patient ID: 365
Unique drugs: 122

Patients with both drug AND expression data: 318


In [33]:
# finalize the expression matrix to patient-level & save processed data

expr['cbio_patient_id'] = expr.index.str.extract(r'(amd_ohsu_2022_\d+)')[0].values

expr_patient = expr.groupby('cbio_patient_id').mean(numeric_only=True)
print("expression matrix after collapsing to patient level:", expr_patient.shape)

final_patient_ids = sorted(final_patients)
expr_final = expr_patient.loc[expr_patient.index.isin(final_patient_ids)]
print("expression matrix for modellable cohort:", expr_final.shape)

drug_final = drug_with_patient[drug_with_patient['cbio_patient_id'].isin(final_patient_ids)].copy()
print("Drug response rows for modellable cohort:", len(drug_final))
print("Unique patients:", drug_final['cbio_patient_id'].nunique())
print("Unique drugs:", drug_final['inhibitor'].nunique())

PROC_DIR = Path("../data/processed")
PROC_DIR.mkdir(exist_ok=True)
expr_final.to_csv(PROC_DIR / "expression_final.csv")
drug_final.to_csv(PROC_DIR / "drug_final.csv", index=False)
print("\nsaved to data/processed/")

expression matrix after collapsing to patient level: (0, 22834)
expression matrix for modellable cohort: (0, 22834)
Drug response rows for modellable cohort: 34764
Unique patients: 318
Unique drugs: 122

saved to data/processed/


In [35]:
# fix the expression collapse

expr_reset = expr.copy()
expr_reset['cbio_patient_id'] = expr_reset.index.str.extract(r'(aml_ohsu_2022_\d+)')[0].values

print("Null patient IDs:", expr_reset['cbio_patient_id'].isnull().sum())
print("Sample:", expr_reset['cbio_patient_id'].head(3).tolist())

expr_patient = expr_reset.groupby('cbio_patient_id').mean(numeric_only=True)
print("After groupby:", expr_patient.shape)

expr_final = expr_patient.loc[expr_patient.index.isin(final_patient_ids)]
print("After filtering to modellable cohort:", expr_final.shape)

expr_final.to_csv(PROC_DIR / "expression_final.csv")
print("saved.")

Null patient IDs: 0
Sample: ['aml_ohsu_2022_2157', 'aml_ohsu_2022_2269', 'aml_ohsu_2022_2313']
After groupby: (613, 22834)
After filtering to modellable cohort: (318, 22834)
saved.
